# YOLOv8-Seg — Training Pipeline

Full training and Optuna HPO pipeline for the YOLOv8-Seg baseline.


In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path
import yaml
import numpy as np
import torch
from ultralytics import YOLO
from PIL import Image
import cv2
import re

base_dir = "/home/zera/Desktop/data_fold/folds/folds"
output_dir = "/home/zera/Desktop/data_fold/yolo_folds_clean"
runs_dir = "/home/zera/Desktop/data_fold/runs/segment"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(runs_dir, exist_ok=True)
print("Environment ready.")

Environment ready.


### 1. Veri Hazırlama (Temiz Verileri Doğru Eşleme - Veri Sızıntısı Giderilmiş)

In [ ]:
print("YOLO Veri setleri hazırlanıyor (sadece temiz resimler + maske label'ları)...\n")
for fold_idx in range(1, 6):
    fold_name = f"FOLD_{fold_idx}"
    fold_path = os.path.join(base_dir, fold_name)
    yolo_fold_dir = os.path.join(output_dir, fold_name)
    
    for split in ['train', 'val']:
        os.makedirs(os.path.join(yolo_fold_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(yolo_fold_dir, 'labels', split), exist_ok=True)

    def process_split(csv_file, split_name):
        df = pd.read_csv(os.path.join(fold_path, csv_file))
        for _, row in df.iterrows():
            image_path_str = str(row['image_path']).replace('\\', '/')
            image_filename = os.path.basename(image_path_str)
            label_filename = image_filename.replace('.png', '.txt').replace('.jpg', '.txt')
            
            win_path_parts = str(row['image_path']).split("\\")
            if "train_val" in win_path_parts:
                idx = win_path_parts.index("train_val")
                rel_path = "/".join(win_path_parts[idx:])
                src_img = Path(fold_path).parent / rel_path
            else:
                src_img = Path(fold_path) / "train_val" / "usg" / image_filename
            if not src_img.exists():
                src_img = Path(fold_path) / "train_val" / "darkened" / image_filename

            src_lbl = Path(fold_path) / "train_val" / "darkened_plgn" / label_filename
            
            dst_img = os.path.join(yolo_fold_dir, 'images', split_name, image_filename)
            dst_lbl = os.path.join(yolo_fold_dir, 'labels', split_name, label_filename)
            
            if src_img.exists() and not os.path.exists(dst_img):
                os.symlink(src_img, dst_img)
            if src_lbl.exists() and not os.path.exists(dst_lbl):
                os.symlink(src_lbl, dst_lbl)

    process_split("train.csv", "train")
    process_split("val.csv", "val")

    dataset_yaml = {
        'path': yolo_fold_dir,
        'train': 'images/train',
        'val': 'images/val',
        'nc': 2,
        'names': ['control', 'arthritis']
    }
    with open(os.path.join(yolo_fold_dir, 'dataset.yaml'), 'w') as f:
        yaml.dump(dataset_yaml, f, sort_keys=False)
        
print("Tüm foldlar yeni, temiz veri seti üzerinden kullanılmaya hazır.")

YOLO Veri setleri hazırlanıyor (sadece temiz resimler + maske label'ları)...

Tüm foldlar yeni, temiz veri seti üzerinden kullanılmaya hazır.


### 2. Tıbbi Veri İçin Gelişmiş YOLOv8 Eğitimi
Ultrason verileri yoğun gürültülü (noisy) olduğu için varsayılan YOLO augmentasyonları (örn: Mosaic) dokuları bozarak modelin kafasını karıştırır. Bu nedenle medikal hedeflere yönelik özel optimizasyonlar uygulanmıştır.

In [ ]:
model_path = "yolov8m-seg.pt"  # yolov8 Medium modeli (daha karmaşık tıbbi dokular için)
epochs = 100

for fold_idx in range(1, 6):
    fold_name = f"FOLD_{fold_idx}"
    print(f"\n[{fold_name}] Eğitim Başlıyor...")
    yaml_path = os.path.join(output_dir, fold_name, "dataset.yaml")
    
    model = YOLO(model_path)
    res = model.train(
        data=yaml_path,
        epochs=epochs,
        patience=20,          # Overfit'i engellemek için early stopping
        batch=16,
        imgsz=640,
        project=runs_dir,
        name=f"train_{fold_name}_medikal", # Klasör isimleri çakışmasın diye değiştirildi
        exist_ok=True,
        workers=8,
        
        # --- Tıbbi Görüntü (USG) Optimizasyonları ---
        optimizer='AdamW',    # Medikal verilerde SGD'den daha stabil davranır
        lr0=0.001,            # Daha yavaş ve istikrarlı öğrenme oranı
        mosaic=0.0,           # Mosaic medikal görüntülerde lezyonları ikiye böler, sıfırlandı
        mixup=0.0,            # Fiziksel yapıyı bozmasını engellemek için sıfırlandı
        degrees=15.0,         # Kist ve lezyonlar ufak rotasyonlarla her açıda bulunmaya alıştırılır
        translate=0.1,        # Görüntü üzerinde gezinme
        scale=0.5,            # Büyütme küçültme ile farklı boyutlardaki kistleri genelleme
        fliplr=0.5,           # Yatay çevirme
        flipud=0.2            # Dikey çevirme (USG problarının farklı tutuş açıları için faydalıdır)
    )
    print(f"[{fold_name}] Eğitimi tamamlandı.")



[FOLD_1] Eğitim Başlıyor...
Ultralytics 8.4.14 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32101MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/zera/Desktop/data_fold/yolo_folds_clean/FOLD_1/dataset.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=train_FOLD_1_medikal, nbs=64

### 3. Test Seti Üzerinde Kapsamlı Değerlendirme, Makale Metrikleri & Görselleştirme

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, roc_curve, precision_recall_curve, auc, accuracy_score
import cv2
import re
from ultralytics import YOLO

def dice_iou_pixel(gt, pr):
    gt = (gt > 0).astype(np.uint8)
    pr = (pr > 0).astype(np.uint8)
    inter = int((gt & pr).sum())
    gs = int(gt.sum())
    ps = int(pr.sum())
    
    tp = inter
    fn = gs - tp
    fp = ps - tp
    tn = gt.size - tp - fp - fn
    
    if gs == 0 and ps == 0: return 1.0, 1.0, 1.0, 1.0
    
    dice = (2.0 * inter / (gs + ps)) if (gs + ps) > 0 else 0.0
    union = (gs + ps - inter)
    iou = (inter / union) if union > 0 else 0.0
    pix = float((gt == pr).mean())
    
    sens = tp / gs if gs > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    if gs == 0:
        auc_val = spec
    elif (tn+fp) == 0:
        auc_val = sens
    else:
        auc_val = 0.5 * (sens + spec)
        
    return float(dice), float(iou), float(pix), float(auc_val)

def get_confusion_matrix_vals(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 1:
        if y_true[0] == 0:
            tn, fp, fn, tp = cm[0,0], 0, 0, 0
        else:
            tn, fp, fn, tp = 0, 0, 0, cm[0,0]
    else:
        tn, fp, fn, tp = cm.ravel() if len(cm.ravel())==4 else (0,0,0,0)
    return tn, fp, fn, tp

def extract_patient_id(filename):
    s = filename.upper().replace('-', '_')
    match = re.search(r'(?:IMAGE_)?(\d{2,3})_[LR]_[CP]', s)
    if match:
        return match.group(1)
    
    parts = s.split('_')
    for p in parts:
        if p.isdigit() and len(p) in (2, 3):
            return p
    return "Unknown"

base_dir = "/home/zera/Desktop/data_fold/folds/folds"
runs_dir = "/home/zera/Desktop/data_fold/runs/segment"

print("\n=== TEST SET PERFORMANCE METRICS ===")
all_folds_metrics = []

# GLOBAL BİRİKTİRİCİLER (Yeni Hücre İçin)
global_gt_areas = []
global_pr_areas = []
global_y_true_pat = {}
global_y_pred_pat = {}

box_dices_all = []
box_accs_all = []
box_f1s_all = []
box_aucs_all = []

for fold_idx in range(1, 6):
    fold_name = f"FOLD_{fold_idx}"
    fold_dir = Path(base_dir) / fold_name
    
    images_dir = fold_dir / "test" / "darkened_plgn"
    roi_dir = fold_dir / "test" / "ROI"
    
    best_pt = Path(runs_dir) / f"train_{fold_name}_medikal" / "weights" / "best.pt"
    if not best_pt.exists():
        print(f"[{fold_name}] Not trained yet, skipping...")
        continue
        
    output_vis_dir = Path(runs_dir) / f"{fold_name}_test_results"
    os.makedirs(output_vis_dir, exist_ok=True)
        
    model = YOLO(str(best_pt))
    res_list = model.predict(source=str(images_dir), imgsz=640, conf=0.01, verbose=False)
    
    seg_dices, seg_ious, seg_pixs, seg_aucs = [], [], [], []
    patient_preds = {} 
    
    gt_areas = []
    pr_areas = []
    
    slice_y_score = []
    slice_y_true = []
    
    for r in res_list:
        img_path = Path(r.path)
        s = img_path.stem.upper()
        
        gt_cls = 1 if ("_P_" in s or "-P" in s or s.endswith("_P")) else 0 if ("_C_" in s or "-C" in s or s.endswith("_C")) else None
        pat_id = extract_patient_id(img_path.name)
        
        score = 0.0
        b = r.boxes
        if b is not None and getattr(b, "conf", None) is not None and len(b) > 0:
            confs = b.conf.detach().cpu().numpy().astype(float)
            clses = b.cls.detach().cpu().numpy().astype(int)
            m = (clses == 1) 
            if m.any(): score = float(np.max(confs[m]))
            
        if gt_cls is not None:
            slice_y_true.append(gt_cls)
            slice_y_score.append(score)
            if pat_id not in patient_preds:
                patient_preds[pat_id] = {'scores': [], 'gt_cls': gt_cls}
            patient_preds[pat_id]['scores'].append(score)
            
        roi_img_path = roi_dir / img_path.name.replace("image_", "mask_")
        pr_mask = np.zeros((640, 640), dtype=np.uint8) 
        gt_mask = np.zeros((640, 640), dtype=np.uint8)
        
        if roi_img_path.exists():
            im = Image.open(roi_img_path).convert("L")
            gt_mask = (np.array(im) > 0).astype(np.uint8)
            pr_mask = np.zeros(gt_mask.shape, dtype=np.uint8)
            
            ms = r.masks
            if ms is not None and ms.data is not None:
                m_data = ms.data.detach().cpu().numpy()
                if len(m_data) > 0:
                    keep = np.ones((m_data.shape[0],), dtype=bool)
                    if b is not None and getattr(b, "conf", None) is not None and len(b) == len(m_data):
                        keep &= (b.conf.detach().cpu().numpy().astype(float) >= 0.05)
                    if keep.any():
                        union = (m_data[keep] > 0.5).any(axis=0).astype(np.uint8)
                        union_im = Image.fromarray(union * 255).resize((gt_mask.shape[1], gt_mask.shape[0]), resample=Image.NEAREST)
                        pr_mask = (np.array(union_im) > 0).astype(np.uint8)
            
            d, i, pix, auc_v = dice_iou_pixel(gt_mask, pr_mask)
            seg_dices.append(d)
            seg_ious.append(i)
            seg_pixs.append(pix)
            seg_aucs.append(auc_v)
            
            gt_areas.append(np.sum(gt_mask))
            
            pr_areas.append(np.sum(pr_mask))
            
            # --- YENİ EKLENEN: SADECE DIŞ ÇİZGİLERDEN OLUŞAN (İÇİ BOŞ) OVERLAY ÇİZİMİ ---
            overlay_dir = output_vis_dir / "overlays"
            os.makedirs(overlay_dir, exist_ok=True)
            
            # Orijinal görüntüyü yükle
            orig_img = cv2.imread(str(img_path))
            if orig_img is not None:
                orig_img = cv2.resize(orig_img, (gt_mask.shape[1], gt_mask.shape[0]))
                
                # Sadece kırmızı ve beyaz kullanılacak. 
                # Ground Truth -> Beyaz (255, 255, 255)
                # Prediction -> Kırmızı (0, 0, 255) (Opencv BGR formatında Kırmızı = (0, 0, 255))
                
                gt_contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                pr_contours, _ = cv2.findContours(pr_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                # İçini doldurmadan sadece çerçeveleri çiziyoruz (thickness=2)
                cv2.drawContours(orig_img, gt_contours, -1, (255, 255, 255), 2)
                cv2.drawContours(orig_img, pr_contours, -1, (0, 0, 255), 2)
                
                # Sağ Üst Köşeye Legend (Açıklama) Ekleme
                font = cv2.FONT_HERSHEY_SIMPLEX
                # Arka plan kutusu (daha okunaklı olması için)
                cv2.rectangle(orig_img, (orig_img.shape[1] - 220, 10), (orig_img.shape[1] - 10, 70), (0, 0, 0), -1)
                # Yazılar
                cv2.putText(orig_img, "White: Ground Truth", (orig_img.shape[1] - 210, 35), font, 0.6, (255, 255, 255), 2)
                cv2.putText(orig_img, "Red: Prediction", (orig_img.shape[1] - 210, 60), font, 0.6, (0, 0, 255), 2)
                
                cv2.imwrite(str(overlay_dir / f"{s}_overlay.png"), orig_img)


    box_dices_all.append([x*100.0 for x in seg_dices])
    box_accs_all.append([x*100.0 for x in seg_pixs])
    box_f1s_all.append([x*100.0 for x in seg_dices])
    box_aucs_all.append([x*100.0 for x in seg_aucs])

    # Alanları global listeye ekle
    global_gt_areas.extend(gt_areas)
    global_pr_areas.extend(pr_areas)

    pix_mean = np.mean(seg_pixs) * 100 if seg_pixs else 0
    d_mean = np.mean(seg_dices) * 100 if seg_dices else 0
    i_mean = np.mean(seg_ious) * 100 if seg_ious else 0
    
    # -------------------------------------------------------------
    # SLICE-LEVEL CLASSIFICATION (Kesit Bazlı Başarım - Matris için referans)
    # -------------------------------------------------------------
    sl_y_true_arr = np.array(slice_y_true)
    sl_y_score_arr = np.array(slice_y_score)
    sl_y_pred_arr = (sl_y_score_arr >= 0.10).astype(int) 
    sl_acc = accuracy_score(sl_y_true_arr, sl_y_pred_arr) * 100 if len(sl_y_true_arr) > 0 else 0
    
    # -------------------------------------------------------------
    # HASTA BAZLI (PATIENT-LEVEL) AGGREGATION
    # -------------------------------------------------------------
    y_true_pat = []
    y_score_pat = []
    for pat_id, data in patient_preds.items():
        y_true_pat.append(data['gt_cls'])
        patient_avg_score = np.mean(data['scores']) if data['scores'] else 0.0
        y_score_pat.append(patient_avg_score)
        
    y_true_arr = np.array(y_true_pat)
    y_score_arr = np.array(y_score_pat)
    y_pred_arr = (y_score_arr >= 0.05).astype(int) 
    
    # Matris verilerini global sözlüğe ekle (Grid çizimi için)
    global_y_true_pat[fold_name] = y_true_arr
    global_y_pred_pat[fold_name] = y_pred_arr

    acc = accuracy_score(y_true_arr, y_pred_arr) * 100 if len(y_true_arr) > 0 else 0
    f1 = f1_score(y_true_arr, y_pred_arr) if len(y_true_arr) > 0 else 0
    
    print(f"\n--- {fold_name} RESULTS ---")
    print(f"SEGMENTATION -> Dice: {d_mean:.2f}% | IoU: {i_mean:.2f}% | Pixel Acc: {pix_mean:.2f}%")
    print(f"SLICE-LEVEL CLASSIFICATION -> Accuracy: {sl_acc:.2f}%")
    print(f"PATIENT-LEVEL CLASSIFICATION (N={len(y_true_arr)}) -> Accuracy: {acc:.2f}% | F1 Score: {f1:.4f}")
    
    roc_auc_val, pr_auc_val = 0.0, 0.0
    if len(y_true_arr) > 0:
        tn, fp, fn, tp = get_confusion_matrix_vals(y_true_arr, y_pred_arr)
        print(f"Patient Matrix: -> TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")
        
        if len(np.unique(y_true_arr)) >= 2:
            fpr, tpr, _ = roc_curve(y_true_arr, y_score_arr)
            roc_auc_val = auc(fpr, tpr)
            precision, recall, _ = precision_recall_curve(y_true_arr, y_score_arr)
            pr_auc_val = auc(recall, precision)
            print(f"Patient Curve: -> ROC-AUC: {roc_auc_val:.4f} | PR-AUC: {pr_auc_val:.4f}")
        
        s_tn, s_fp, s_fn, s_tp = get_confusion_matrix_vals(sl_y_true_arr, sl_y_pred_arr)
        print(f"Slice Matrix: -> TP: {s_tp}, TN: {s_tn}, FP: {s_fp}, FN: {s_fn}")
        
    all_folds_metrics.append({
        'fold': fold_name,
        'dice': d_mean, 'iou': i_mean, 'pix_acc': pix_mean,
        'acc': acc, 'f1': f1, 'roc_auc': roc_auc_val, 'pr_auc': pr_auc_val,
        'slice_acc': sl_acc
    })

if len(all_folds_metrics) > 0:
    print("\n====== 5-FOLD CROSS VALIDATION AVERAGES ======")
    m_dice = np.mean([x['dice'] for x in all_folds_metrics])
    m_iou = np.mean([x['iou'] for x in all_folds_metrics])
    m_pix = np.mean([x['pix_acc'] for x in all_folds_metrics])
    m_sl_acc = np.mean([x['slice_acc'] for x in all_folds_metrics])
    m_acc = np.mean([x['acc'] for x in all_folds_metrics])
    m_f1 = np.mean([x['f1'] for x in all_folds_metrics])
    m_roc = np.mean([x['roc_auc'] for x in all_folds_metrics])
    m_pr = np.mean([x['pr_auc'] for x in all_folds_metrics])
    
    print(f"ORT. SEGMENTATION -> Dice: {m_dice:.2f}% | IoU: {m_iou:.2f}%")
    print(f"AVG SLICE CLASSIFICATION -> Accuracy: {m_sl_acc:.2f}%")
    print(f"AVG PATIENT CLASSIFICATION -> Accuracy: {m_acc:.2f}% | F1 Score: {m_f1:.4f} | ROC-AUC: {m_roc:.4f} | PR-AUC: {m_pr:.4f}")


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from sklearn.metrics import confusion_matrix

output_vis_dir = "/home/zera/Desktop/data_fold/runs/segment/Global_Results"
os.makedirs(output_vis_dir, exist_ok=True)

print("Generating global visuals, please wait...")

# 1. GLOBAL BLAND-ALTMAN PLOT (Tüm Foldlar Ortalama Alan)
gt = np.asarray(global_gt_areas, dtype=np.float32)
pr = np.asarray(global_pr_areas, dtype=np.float32)

mask = (gt > 0) | (pr > 0)
gt_f = gt[mask]
pr_f = pr[mask]

if len(gt_f) > 0:
    mean = np.mean([gt_f, pr_f], axis=0)
    diff = pr_f - gt_f                  
    md = np.mean(diff)              
    sd = np.std(diff, axis=0)       

    plt.figure(figsize=(10,6))
    plt.scatter(mean, diff, alpha=0.5, color='royalblue')
    plt.axhline(md, color='red', linestyle='--', label=f'Mean Difference: {md:.2f}')
    plt.axhline(md + 1.96*sd, color='gray', linestyle='--', label=f'+1.96 SD: {md + 1.96*sd:.2f}')
    plt.axhline(md - 1.96*sd, color='gray', linestyle='--', label=f'-1.96 SD: {md - 1.96*sd:.2f}')
    plt.title('Global Bland-Altman Plot (All Folds)')
    plt.xlabel('Mean Area (pixels)')
    plt.ylabel('Difference (Predicted - Ground Truth)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_vis_dir, 'global_bland_altman.png'), dpi=300)
    plt.show()

# 2. CONFUSION MATRIX GRID (Fold 1-5 + Toplam)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

all_y_true = []
all_y_pred = []

ordered_folds = [f"FOLD_{i}" for i in range(1, 6)]

for idx, f_name in enumerate(ordered_folds):
    if f_name in global_y_true_pat:
        y_true = global_y_true_pat[f_name]
        y_pred = global_y_pred_pat[f_name]
        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)
        
        cm = confusion_matrix(y_true, y_pred)
        if cm.size == 1:
            cm = np.array([[cm[0,0], 0], [0, 0]]) if y_true[0] == 0 else np.array([[0, 0], [0, cm[0,0]]])
        elif cm.shape != (2, 2):
            cm = np.pad(cm, ((0, 2-cm.shape[0]), (0, 2-cm.shape[1])), 'constant')
            
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={'size': 20, 'weight': 'bold'}, 
                    xticklabels=['Control', 'Arthritis'], 
                    yticklabels=['Control', 'Arthritis'], 
                    ax=axes[idx], cbar=False)
        axes[idx].set_title(f'{f_name} (Patient-Level)', fontsize=16, fontweight='bold')
        axes[idx].set_ylabel('True Label', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Predicted Label', fontsize=14, fontweight='bold')
    else:
        axes[idx].axis('off')

# Son hücreye "Tüm Foldların Toplamı"nı çizdirelim
if len(all_y_true) > 0:
    cm_total = confusion_matrix(all_y_true, all_y_pred)
    if cm_total.size == 1:
        cm_total = np.array([[cm_total[0,0], 0], [0, 0]]) if all_y_true[0] == 0 else np.array([[0, 0], [0, cm_total[0,0]]])
    elif cm_total.shape != (2, 2):
        cm_total = np.pad(cm_total, ((0, 2-cm_total.shape[0]), (0, 2-cm_total.shape[1])), 'constant')
        
    sns.heatmap(cm_total, annot=True, fmt='d', cmap='Greens', annot_kws={'size': 24, 'weight': 'bold'}, 
                xticklabels=['Control', 'Arthritis'], 
                yticklabels=['Control', 'Arthritis'], 
                ax=axes[5])
    axes[5].set_title('ALL FOLDS TOTAL (Global)', fontsize=18, fontweight='bold')
    axes[5].set_ylabel('True Label', fontsize=16, fontweight='bold')
    axes[5].set_xlabel('Predicted Label', fontsize=16, fontweight='bold')
else:
    axes[5].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(output_vis_dir, 'global_confusion_matrix_grid.png'), dpi=300)
plt.show()

print(f"Visuals successfully saved to: {output_vis_dir}")



# 3. BOX PLOT GRİDİ (F1..F5 İçin Dağılımlar)
print("Drawing boxplots...")
fig_bp, axes_bp = plt.subplots(2, 2, figsize=(14, 10))
bp_labels = ['F1', 'F2', 'F3', 'F4', 'F5']

def do_boxplot(ax, data, title, ylabel):
    boxprops = dict(facecolor='lightblue', alpha=0.7)
    medianprops = dict(color='darkorange', linewidth=2)
    meanpointprops = dict(marker='^', markeredgecolor='green', markerfacecolor='green')
    
    ax.boxplot(data, tick_labels=bp_labels, patch_artist=True, showmeans=True, 
               boxprops=boxprops, medianprops=medianprops, meanprops=meanpointprops)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Fold')
    ax.grid(axis='y', linestyle='-', alpha=0.3)
    ax.set_ylim(-5, 105)

if len(box_dices_all) == 5:
    do_boxplot(axes_bp[0,0], box_dices_all, 'Test mDice (%)', 'mDice (%)')
    do_boxplot(axes_bp[0,1], box_accs_all, 'Val Accuracy (%)', 'Accuracy (%)')
    do_boxplot(axes_bp[1,0], box_f1s_all, 'Val F1-Score (%)', 'F1-Score (%)')
    do_boxplot(axes_bp[1,1], box_aucs_all, 'Val ROC-AUC (%)', 'ROC-AUC (%)')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_vis_dir, 'folds_boxplots.png'), dpi=300)
    plt.show()
    print(f"Boxplots successfully saved: {os.path.join(output_vis_dir, 'folds_boxplots.png')}")
else:
    print("Fold count incomplete, skipping boxplots.")



=== TEST SET PERFORMANCE METRICS ===

--- FOLD_1 RESULTS ---
SEGMENTATION -> Dice: 79.21% | IoU: 67.85% | Pixel Acc: 98.95%
SLICE-LEVEL CLASSIFICATION -> Accuracy: 73.85%
PATIENT-LEVEL CLASSIFICATION (N=11) -> Accuracy: 54.55% | F1 Score: 0.5455
Patient Matrix: -> TP: 3, TN: 3, FP: 5, FN: 0
Patient Curve: -> ROC-AUC: 0.7917 | PR-AUC: 0.4278
Slice Matrix: -> TP: 40, TN: 56, FP: 24, FN: 10

--- FOLD_2 RESULTS ---
SEGMENTATION -> Dice: 76.47% | IoU: 65.35% | Pixel Acc: 98.67%
SLICE-LEVEL CLASSIFICATION -> Accuracy: 79.23%
PATIENT-LEVEL CLASSIFICATION (N=10) -> Accuracy: 90.00% | F1 Score: 0.9091
Patient Matrix: -> TP: 5, TN: 4, FP: 1, FN: 0
Patient Curve: -> ROC-AUC: 0.9600 | PR-AUC: 0.9633
Slice Matrix: -> TP: 45, TN: 58, FP: 2, FN: 25

--- FOLD_3 RESULTS ---
SEGMENTATION -> Dice: 75.21% | IoU: 62.73% | Pixel Acc: 98.79%
SLICE-LEVEL CLASSIFICATION -> Accuracy: 77.86%
PATIENT-LEVEL CLASSIFICATION (N=13) -> Accuracy: 76.92% | F1 Score: 0.8000
Patient Matrix: -> TP: 6, TN: 4, FP: 3, FN: 0


<Figure size 1000x600 with 1 Axes>

<Figure size 1500x900 with 7 Axes>

Visuals successfully saved to: /home/zera/Desktop/data_fold/runs/segment/Global_Results
Drawing boxplots...


<Figure size 1400x1000 with 4 Axes>

Boxplots successfully saved: /home/zera/Desktop/data_fold/runs/segment/Global_Results/folds_boxplots.png
